# TSForecasting: Builder Pattern and Feature Selection

This notebook demonstrates the fluent builder pattern for pipeline configuration and the `TreeBasedFeatureSelector` for dimensionality reduction.

**Topics covered:**
- `TSForecastingBuilder` fluent pipeline configuration
- Method chaining configuration
- `TreeBasedFeatureSelector` usage (multi-horizon)
- Feature importance analysis by horizon
- Integration with the pipeline

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=Warning)

from tsforecasting import (
    Processing,
    TimeSeriesDatasetGenerator,
    TreeBasedFeatureSelector,
    TSForecastingBuilder,
    model_configurations,
)

---
## 1. Builder Pattern

The `TSForecastingBuilder` configures the pipeline through chained `with_*` calls and returns a ready-to-fit `TSForecasting` instance with `build()`.

In [ ]:
data = TimeSeriesDatasetGenerator.generate(
    n_samples=250,
    granularity="1mo",
    patterns=["trend", "seasonal"],
    random_state=42,
)

print(f"Data shape: {data.shape}")
data.head()

### 1.1 Basic Builder Usage

In [ ]:
# Fluent configuration with method chaining
pipeline = (
    TSForecastingBuilder()
    .with_train_size(0.80)
    .with_lags(12)
    .with_horizon(6)
    .with_sliding_size(6)
    .with_models(["RandomForest", "XGBoost"])
    .with_granularity("1mo")
    .with_metric("MAE")
    .build()
)

print("Pipeline built with builder pattern:")
print(f"  train_size: {pipeline.train_size}")
print(f"  lags: {pipeline.lags}")
print(f"  horizon: {pipeline.horizon}")

### 1.2 Full Builder Configuration

In [ ]:
# Get hyperparameters
hparams = model_configurations()
hparams["RandomForest"]["n_estimators"] = 30
hparams["XGBoost"]["n_estimators"] = 30

# Full configuration
pipeline = (
    TSForecastingBuilder()
    # Data split
    .with_train_size(0.80)
    # Time series parameters
    .with_lags(10)
    .with_horizon(5)
    .with_sliding_size(5)
    # Model selection
    .with_models(["RandomForest", "GBR", "XGBoost"])
    # Hyperparameters
    .with_hparameters(hparams)
    # Evaluation settings
    .with_granularity("1mo")
    .with_metric("MAE")
    # Preprocessing options
    .with_preprocessing(
        scaler="standard",  # Options: "standard", "minmax", "robust"
        datetime_features=True,  # Add date-based features
    )
    # Output
    .with_verbose(False)  # Silence the evaluation progress
    # Build the pipeline
    .build()
)

print("Full pipeline configuration:")
print(f"  train_size: {pipeline.train_size}")
print(f"  lags: {pipeline.lags}")
print(f"  horizon: {pipeline.horizon}")
print(f"  sliding_size: {pipeline.sliding_size}")
print(f"  granularity: {pipeline.granularity}")
print(f"  metric: {pipeline.metric}")

In [ ]:
# Fit the pipeline
pipeline.fit_forecast(data)

print(f"Best model: {pipeline.selected_model}")
pipeline.history().leaderboard

### 1.3 Builder Methods Reference

| Method | Description |
|--------|-------------|
| `.with_train_size(float)` | Initial train proportion, 0.3 up to (excluding) 1.0 |
| `.with_lags(int)` | Number of lagged features |
| `.with_horizon(int)` | Forecast horizon |
| `.with_sliding_size(int)` | Window expansion size |
| `.with_models(list)` | List of model names (case-insensitive) |
| `.with_hparameters(dict)` | Model hyperparameters |
| `.with_granularity(str)` | `"1m"`, `"30m"`, `"1h"`, `"1d"`, `"1wk"`, `"1mo"` |
| `.with_metric(str)` | `"MAE"`, `"MAPE"`, `"MSE"` |
| `.with_preprocessing(...)` | Scaler and feature options |
| `.with_scaler(str)` | `"standard"`, `"minmax"`, `"robust"` |
| `.with_datetime_features(bool)` | Add/skip date-based features |
| `.with_verbose(bool)` | Print/silence evaluation progress |
| `.build()` | Create the `TSForecasting` instance |

Every setter validates its value: an out-of-range number or an unknown name raises `ConfigurationError` (`ModelNotFoundError` for models).

In [ ]:
from tsforecasting import ConfigurationError

try:
    TSForecastingBuilder().with_train_size(5)
except ConfigurationError as error:
    print(f"Rejected: {error}")

---
## 2. Feature Selection (Multi-Horizon)

`TreeBasedFeatureSelector` computes feature importances across **all** forecast horizons, using a weighted aggregation where closer horizons (h1, h2, ...) receive higher weight than distant ones.

Weight formula: `w_h = decay^(h-1)`, normalized to sum to 1.

With the default `decay=0.8` and 5 horizons: h1=0.30, h2=0.24, h3=0.19, h4=0.15, h5=0.12.

### 2.1 Prepare Data for Feature Selection

In [ ]:
# Generate larger dataset
data = TimeSeriesDatasetGenerator.generate(
    n_samples=300,
    granularity="1mo",
    patterns=["trend", "seasonal", "regime_change"],
    random_state=42,
)

# Transform to supervised format
processor = Processing()
timeseries = processor.make_timeseries(
    dataset=data,
    window_size=15,  # More lags = more features
    horizon=5,
    datetime_engineering=True,
)

# Remove rows with incomplete targets (the last `horizon` rows)
target_cols = [c for c in timeseries.columns if c.startswith("y_horizon_")]
valid_mask = ~timeseries[target_cols].isna().any(axis=1)
timeseries = timeseries[valid_mask].copy()

# Separate features and targets (ALL horizons)
feature_cols = [c for c in timeseries.columns if not c.startswith("y_horizon_") and c != "Date"]
X = timeseries[feature_cols]
y = timeseries[target_cols]  # All horizons as DataFrame

print(f"Features shape: {X.shape}")
print(f"Targets shape: {y.shape} (multi-horizon)")
print(f"\nTarget columns: {list(y.columns)}")
print(f"\nFeature columns ({len(feature_cols)}):")
for col in feature_cols[:10]:
    print(f"  {col}")
if len(feature_cols) > 10:
    print(f"  ... and {len(feature_cols) - 10} more")

### 2.2 TreeBasedFeatureSelector Configuration

Available algorithms: `"RandomForest"`, `"ExtraTrees"`, `"GBR"`.

| Parameter | Description |
|-----------|-------------|
| `algorithm` | Tree model for importance calculation |
| `n_estimators` | Number of trees |
| `relevance_threshold` | Cumulative importance to retain (0.5 to 1.0) |
| `horizon_decay` | Weight decay for horizons (0.5 to 1.0); lower = more emphasis on near-term horizons |
| `random_state` | Reproducibility seed |

In [ ]:
selector = TreeBasedFeatureSelector(
    algorithm="ExtraTrees",
    n_estimators=100,
    relevance_threshold=0.95,
    horizon_decay=0.8,  # h1 weighted ~2.4x more than h5
    random_state=42,
)

### 2.3 Fit and Analyze Feature Importance (Multi-Horizon)

In [ ]:
# Fit selector with ALL horizons
selector.fit(X, y)

# Horizon weights
print(f"Horizon weights (decay={selector.get_selection_summary()['horizon_decay']}):")
for i, w in enumerate(selector.horizon_weights):
    print(f"  Horizon {i + 1}: {w:.4f} ({w * 100:.1f}%)")

In [ ]:
# Aggregated importances (weighted mean across horizons)
selector.feature_importances.head(15)

In [ ]:
# Side-by-side comparison: per-horizon importances + weighted mean
selector.get_importance_comparison().head(10)

In [ ]:
# Selected features
selected = selector.selected_features
print(f"Selected features: {len(selected)} / {len(feature_cols)}")
print(f"  Features retained: {selected}")

# Selection summary
print("\nSelection Summary:")
for key, value in selector.get_selection_summary().items():
    print(f"  {key}: {value}")

### 2.4 Transform Data

In [ ]:
# Transform to selected features only
X_selected = selector.transform(X)

print(f"Original shape: {X.shape}")
print(f"Reduced shape: {X_selected.shape}")
print(f"Dimensionality reduction: {X.shape[1]} -> {X_selected.shape[1]} features")
print(f"Reduction: {100 * (1 - X_selected.shape[1] / X.shape[1]):.1f}%")

### 2.5 Feature Selection Strategies

In [ ]:
# Compare relevance thresholds
for threshold in [0.99, 0.90, 0.80]:
    sel = TreeBasedFeatureSelector(
        algorithm="ExtraTrees",
        relevance_threshold=threshold,
        horizon_decay=0.8,
    )
    sel.fit(X, y)
    print(f"Threshold {threshold:.2f}, decay 0.8: {len(sel.selected_features)} features")

In [ ]:
# Compare decay factors
print("Horizon decay comparison (threshold=0.95):")
for decay in [0.5, 0.7, 0.8, 0.9, 1.0]:
    sel = TreeBasedFeatureSelector(
        algorithm="ExtraTrees",
        relevance_threshold=0.95,
        horizon_decay=decay,
    )
    sel.fit(X, y)
    weights_str = ", ".join([f"h{i + 1}={w:.2f}" for i, w in enumerate(sel.horizon_weights)])
    print(f"  decay={decay}: {len(sel.selected_features)} features (weights: {weights_str})")

In [ ]:
# Compare algorithms
print("Algorithm comparison (threshold=0.95, decay=0.8):")
for algo in ["RandomForest", "ExtraTrees", "GBR"]:
    sel = TreeBasedFeatureSelector(
        algorithm=algo,
        relevance_threshold=0.95,
        horizon_decay=0.8,
    )
    sel.fit(X, y)
    print(f"  {algo}: {len(sel.selected_features)} features")

---
## 3. Integrated Workflow

Combining the builder pattern with forecasting and prediction intervals.

In [ ]:
# Step 1: Generate data
data = TimeSeriesDatasetGenerator.generate(
    n_samples=300,
    granularity="1mo",
    patterns=["trend", "seasonal"],
    random_state=42,
)

# Step 2: Build pipeline with builder
pipeline = (
    TSForecastingBuilder()
    .with_train_size(0.80)
    .with_lags(12)
    .with_horizon(5)
    .with_sliding_size(5)
    .with_models(["RandomForest", "XGBoost"])
    .with_metric("MAE")
    .with_granularity("1mo")
    .with_verbose(False)
    .build()
)

# Step 3: Fit and evaluate
pipeline.fit_forecast(data)

# Step 4: Results
print(f"Best model: {pipeline.selected_model}")
pipeline.history().leaderboard

In [ ]:
# Step 5: Forecast with prediction intervals
forecast = pipeline.forecast(interval_method="ensemble")
forecast[["Date", "y", "y_lower_90", "y_upper_90"]]

---
## 4. Quick Reference

### Builder Pattern

```python
from tsforecasting import TSForecastingBuilder

pipeline = (
    TSForecastingBuilder()
    .with_train_size(0.80)
    .with_lags(12)
    .with_horizon(6)
    .with_models(["RandomForest", "XGBoost"])
    .with_metric("MAE")
    .build()
)
pipeline.fit_forecast(data)
```

### Multi-Horizon Feature Selection

```python
from tsforecasting import TreeBasedFeatureSelector

# Single horizon
selector = TreeBasedFeatureSelector(relevance_threshold=0.95)
selector.fit(X, y_horizon_1)

# Multi-horizon with decay weighting
selector = TreeBasedFeatureSelector(
    algorithm="ExtraTrees",
    relevance_threshold=0.95,
    horizon_decay=0.8,  # h1 weighted more than h5
)
selector.fit(X, y_all_horizons)  # y can be a DataFrame with multiple columns

# Analyze
selector.feature_importances          # Aggregated (weighted mean)
selector.importances_by_horizon       # Per-horizon breakdown
selector.get_importance_comparison()  # Side-by-side comparison
selector.get_selection_summary()      # Full summary with weights

# Transform
X_reduced = selector.transform(X)
```

### Horizon Decay Explained

| Decay | Weights for 5 horizons |
|-------|------------------------|
| `1.0` | Equal weight to all horizons (20% each) |
| `0.8` | h1=30%, h2=24%, h3=19%, h4=15%, h5=12% |
| `0.5` | h1=52%, h2=26%, h3=13%, h4=6%, h5=3% (aggressive near-term focus) |